<h1> Ant Colony Optimization (ACO) </h1>

Fundamental difference from PSO


|          | PSO         | ACO                |
| -------- | ----------- | ------------------ |
| Agent    | Particle    | Ant                |
| Solution | Position    | Path               |
| Memory   | pbest/gbest | Pheromone          |
| Learning | Movement    | Path Reinforcement |
| Type     | Continuous  | Mostly Discrete    |


PSO says:</br>
Move towards a better solution.

ACO says:</br>
Select paths that have been successful in the past more frequently.

### ACO — The Core Idea

ACO is inspired by the behavior of ants.

When searching for food, ants:

Explore various paths.</br>
Deposit pheromones along the paths.</br>
Accumulate more pheromones on better paths.</br>
Are more likely to choose those same paths.

Thus:</br>
Good paths are reinforced over time.

Initialize Pheromones

        ↓

For each Ant:

    Construct Route

        ↓

    Evaluate Route

        ↓

Evaporate Pheromones

        ↓

Deposit New Pheromones

        ↓

Update Best Route

        ↓

Repeat

Two main concepts in ACO:

1) Pheromone

A value exists for each edge:

$\tau_{ij}$

This represents the learned attractiveness of the path between city $i$ and city $j$.

</br>

2) Heuristic Information

Typically for TSP:

$$ \eta_{ij}=\frac{1}{d_{ij}} $$

Meaning:

A shorter path is more attractive.

Next-city selection rule

Probability of selecting city \(j\) from city \(i\):

$$ P_{ij} = \frac{ \tau_{ij}^{\alpha} \eta_{ij}^{\beta} }{ \sum_{k \in allowed} \tau_{ik}^{\alpha} \eta_{ik}^{\beta} } $$

${\alpha}$

Importance of pheromone.

If increased:

Ants rely more on past experience.

${\beta}$

Importance of distance/heuristic.

If increased:

Ants are more inclined toward closer cities.

</br>

*Evaporation*

If we only add pheromone, old paths constantly become stronger.

So, we have:

$$ \tau_{ij} \leftarrow (1-\rho)\tau_{ij} $$

where:

$$ \rho $$

is the evaporation rate.

This step is crucial for preventing:

Premature convergence.

Pheromone Deposit

After constructing a path:

If an ant finds a good path:

$$ \Delta \tau = \frac{Q}{L} $$

where:

$Q$ is the reinforcement constant

$L$ is the path length.

Thus, a shorter path:

$$ L\downarrow $$

results in:

$$ \Delta\tau\uparrow $$

ACO Implementation Steps

1. Defining the TSP Problem
2. Distance Matrix
3. Pheromone Matrix
4. Probabilistic selection of the next city
5. Route construction by an ant
6. Calculating route length
7. Pheromone evaporation
8. Pheromone deposit
9. Complete ACO loop
10. Visualization of the best route
11. Analysis of parameters (${\alpha}$, ${\beta}$, $\rho$)

In [ ]:
import numpy as np

from problems.tsp import (
    TSPProblem,
    create_complete_graph,
)
from src.ACO import (
    initialize_pheromones,
    build_heuristic_matrix,
    construct_ant_route,
    update_pheromones,
)


<h3> Step 1: Creating a sample problem </h3>

In [2]:
cities = [
    [1, 1],
    [2, 5],
    [5, 8],
    [8, 7],
    [9, 3],
    [6, 1],
    [4, 4],
    [7, 5],
]

In [3]:
graph = create_complete_graph(
    len(cities)
)

problem = TSPProblem(
    cities,
    graph,
)

In [4]:
graph

array([[False,  True,  True,  True,  True,  True,  True,  True],
       [ True, False,  True,  True,  True,  True,  True,  True],
       [ True,  True, False,  True,  True,  True,  True,  True],
       [ True,  True,  True, False,  True,  True,  True,  True],
       [ True,  True,  True,  True, False,  True,  True,  True],
       [ True,  True,  True,  True,  True, False,  True,  True],
       [ True,  True,  True,  True,  True,  True, False,  True],
       [ True,  True,  True,  True,  True,  True,  True, False]])

Route Test

In other words, if even a single edge of the route is missing:

$$ RouteLength=\infty $$

and consequently, that route is effectively infeasible.

In [5]:
route = [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
]

print(
    "Route Length:",
    problem.route_length(route)
)

Route Length: 33.235612360547314


There is one important point regarding the next steps: in a random graph, a Hamiltonian cycle might not exist at all. Therefore, when we later run ACO on a constrained graph, we will need to handle the possibilities of **dead-ends** and **infeasible routes** during path construction. However, since we are currently using a complete graph, we do not face this issue.

<h3> Step 2: Creating a sample problem </h3>

In [6]:
pheromones = initialize_pheromones(
    problem.num_cities
)

heuristic = build_heuristic_matrix(
    problem
)

In [7]:
route, feasible = construct_ant_route(
    problem,
    pheromones,
    heuristic,
    alpha=1.0,
    beta=2.0,
)

In [8]:
print(
    "Route:",
    route
)

print(
    "Feasible:",
    feasible
)

print(
    "Length:",
    problem.route_length(
        route
    )
)

Route: [5, 4, 3, 7, 6, 1, 0, 2]
Feasible: True
Length: 34.61950170203129


<h3> Step 3: Pheromone Evaporation & Deposit </h3>

So far, each ant can construct a route. Now, the quality of the routes must influence the pheromone levels.

General idea:

$$ \text{Pheromone Update} = \text{Evaporation} + \text{Deposit} $$


1) Pheromone Evaporation

Formula:

$$ \tau_{ij} \leftarrow (1-\rho)\tau_{ij} $$

where:

$$ 0<\rho<1 $$

is the evaporation rate.

</br>

2) Pheromone Deposit

If an ant finds a good route, the edges of that route must be reinforced.

Common formula:

$$ \Delta\tau = \frac{Q}{L} $$

where:

\(Q\) is the deposit intensity (Q=1)

\(L\) is the route length

Thus, the shorter the route:

$$ L\downarrow \Rightarrow \Delta\tau\uparrow $$

The typical sequence:

Old Pheromones</br>
↓</br>
Evaporation</br>
↓</br>
Deposit New Information

In other words, the old memory fades slightly first, and then the new experience is added.

This helps ensure the colony does not remain permanently bound to old paths.

making routes

In [9]:
routes = []
route_lengths = []

In [10]:
for _ in range(5):

    route, feasible = construct_ant_route(
        problem,
        pheromones,
        heuristic,
        alpha=1.0,
        beta=2.0,
    )

    if feasible:
        length = problem.route_length(
            route
        )
    else:
        length = np.inf

    routes.append(
        route
    )

    route_lengths.append(
        length
    )

In [11]:
print(
    "Before Update:"
)

print(
    pheromones
)

Before Update:
[[1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]]


In [12]:
routes

[[2, 7, 5, 3, 6, 1, 0, 4],
 [6, 7, 1, 2, 4, 3, 5, 0],
 [2, 7, 3, 6, 5, 4, 1, 0],
 [7, 3, 4, 6, 5, 1, 2, 0],
 [0, 1, 6, 7, 3, 4, 5, 2]]

In [ ]:
update_pheromones(
    pheromones,
    routes,
    route_lengths,
    evaporation_rate=0.2,
    q=1.0,
)